# UFC Fight Data — Exploratory Data Analysis

In [ ]:
# Cell 1 — Imports & Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

DATA_DIR = "./data"
print("✅ Ready")

In [ ]:
# Cell 2 — Load Cleaned Data
fights = pd.read_csv(f"{DATA_DIR}/fights_clean.csv", parse_dates=["event_date"])
fighters = pd.read_csv(f"{DATA_DIR}/fighters_clean.csv")

print("FIGHTS DATASET")
print("=" * 50)
print(f"Shape: {fights.shape}")
print(f"Date range: {fights['event_date'].min()} to {fights['event_date'].max()}")
print(f"\nColumn types:\n{fights.dtypes}")
print(f"\nMissing values:\n{fights.isnull().sum()}")
print(f"\nBasic stats:\n{fights.describe()}")

In [ ]:
# Cell 3 — Fight Outcome Distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

method_counts = fights["finish_type"].value_counts()
axes[0].pie(method_counts, labels=method_counts.index, autopct="%1.1f%%",
            startangle=90, colors=sns.color_palette("Set2"))
axes[0].set_title("Fight Finish Methods", fontsize=14, fontweight="bold")

f1_wr = fights["f1_win"].mean()
axes[1].bar(["Fighter 1 Wins", "Fighter 2 Wins"], [f1_wr, 1 - f1_wr],
            color=["#2ecc71", "#e74c3c"])
axes[1].set_ylabel("Proportion")
axes[1].set_title(f"Positional Bias Check\n(F1 win rate: {f1_wr:.3f})", fontsize=14, fontweight="bold")
axes[1].set_ylim(0, 1)

round_counts = fights["round"].value_counts().sort_index()
axes[2].bar(round_counts.index, round_counts.values, color=sns.color_palette("viridis", len(round_counts)))
axes[2].set_xlabel("Round")
axes[2].set_ylabel("Number of Fights")
axes[2].set_title("Fight Ending Round Distribution", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/eda_outcome_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 4 — Fights Over Time
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

fights["year"] = fights["event_date"].dt.year
yearly = fights.groupby("year").size()
axes[0].plot(yearly.index, yearly.values, marker="o", linewidth=2, markersize=6)
axes[0].fill_between(yearly.index, yearly.values, alpha=0.3)
axes[0].set_title("Number of UFC Fights Per Year", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Number of Fights")

finish_by_year = fights.groupby(["year", "finish_type"]).size().unstack(fill_value=0)
finish_pct = finish_by_year.div(finish_by_year.sum(axis=1), axis=0)
finish_pct.plot(kind="area", stacked=True, ax=axes[1], alpha=0.7)
axes[1].set_title("Fight Finish Methods Over Time (Proportion)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Proportion")
axes[1].legend(title="Finish Type", loc="upper right")

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/eda_time_trends.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 5 — Weight Class Analysis
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

wc_counts = fights["weight_class"].value_counts().head(15)
axes[0].barh(range(len(wc_counts)), wc_counts.values, color=sns.color_palette("coolwarm", len(wc_counts)))
axes[0].set_yticks(range(len(wc_counts)))
axes[0].set_yticklabels(wc_counts.index)
axes[0].set_xlabel("Number of Fights")
axes[0].set_title("Fights by Weight Class (Top 15)", fontsize=14, fontweight="bold")
axes[0].invert_yaxis()

ko_rates = fights.groupby("weight_class").apply(
    lambda x: (x["finish_type"] == "KO/TKO").mean()
).sort_values(ascending=True)
ko_rates = ko_rates[ko_rates.index.isin(wc_counts.index)]
ko_rates.plot(kind="barh", ax=axes[1], color=sns.color_palette("Reds_r", len(ko_rates)))
axes[1].set_xlabel("KO/TKO Rate")
axes[1].set_title("KO/TKO Rate by Weight Class", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/eda_weight_classes.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 6 — Striking Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

fights["winner_str_landed"] = np.where(
    fights["f1_win"] == 1, fights["f1_str_landed"], fights["f2_str_landed"]
)
fights["loser_str_landed"] = np.where(
    fights["f1_win"] == 1, fights["f2_str_landed"], fights["f1_str_landed"]
)

axes[0, 0].hist(fights["winner_str_landed"].dropna(), bins=50, alpha=0.6, label="Winner", color="#2ecc71")
axes[0, 0].hist(fights["loser_str_landed"].dropna(), bins=50, alpha=0.6, label="Loser", color="#e74c3c")
axes[0, 0].set_xlabel("Significant Strikes Landed")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].set_title("Strikes Landed: Winner vs Loser", fontsize=13, fontweight="bold")
axes[0, 0].legend()

fights["winner_str_acc"] = np.where(fights["f1_win"] == 1, fights["f1_str_acc"], fights["f2_str_acc"])
fights["loser_str_acc"] = np.where(fights["f1_win"] == 1, fights["f2_str_acc"], fights["f1_str_acc"])

axes[0, 1].hist(fights["winner_str_acc"].dropna(), bins=40, alpha=0.6, label="Winner", color="#2ecc71")
axes[0, 1].hist(fights["loser_str_acc"].dropna(), bins=40, alpha=0.6, label="Loser", color="#e74c3c")
axes[0, 1].set_xlabel("Striking Accuracy")
axes[0, 1].set_title("Striking Accuracy: Winner vs Loser", fontsize=13, fontweight="bold")
axes[0, 1].legend()

kd_diff = fights["f1_kd"].fillna(0) - fights["f2_kd"].fillna(0)
fights["kd_diff"] = kd_diff
kd_win_rate = fights.groupby("kd_diff")["f1_win"].mean()
kd_win_rate = kd_win_rate[(kd_win_rate.index >= -3) & (kd_win_rate.index <= 3)]
axes[1, 0].bar(kd_win_rate.index, kd_win_rate.values, color=sns.color_palette("RdYlGn", len(kd_win_rate)))
axes[1, 0].axhline(y=0.5, color="gray", linestyle="--", alpha=0.7)
axes[1, 0].set_xlabel("Knockdown Differential (F1 - F2)")
axes[1, 0].set_ylabel("F1 Win Rate")
axes[1, 0].set_title("Knockdown Differential vs Win Rate", fontsize=13, fontweight="bold")

fights["str_diff"] = fights["f1_str_landed"].fillna(0) - fights["f2_str_landed"].fillna(0)
fights["str_diff_bin"] = pd.cut(fights["str_diff"], bins=20)
str_win = fights.groupby("str_diff_bin", observed=True)["f1_win"].agg(["mean", "count"])
str_win = str_win[str_win["count"] >= 10]

x_vals = [interval.mid for interval in str_win.index]
axes[1, 1].scatter(x_vals, str_win["mean"], s=str_win["count"], alpha=0.6, c="#3498db")
axes[1, 1].axhline(y=0.5, color="gray", linestyle="--", alpha=0.7)
axes[1, 1].set_xlabel("Strike Differential (F1 - F2)")
axes[1, 1].set_ylabel("F1 Win Probability")
axes[1, 1].set_title("Strike Differential vs Win Probability\n(size = sample count)", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/eda_striking.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 7 — Fighter Physical Attributes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(fighters["height_inches"].dropna(), bins=30, color="#3498db", edgecolor="white")
axes[0].set_xlabel("Height (inches)")
axes[0].set_title("Fighter Height Distribution", fontweight="bold")

axes[1].hist(fighters["reach_inches"].dropna(), bins=30, color="#e67e22", edgecolor="white")
axes[1].set_xlabel("Reach (inches)")
axes[1].set_title("Fighter Reach Distribution", fontweight="bold")

axes[2].hist(fighters["weight_lbs"].dropna(), bins=30, color="#2ecc71", edgecolor="white")
axes[2].set_xlabel("Weight (lbs)")
axes[2].set_title("Fighter Weight Distribution", fontweight="bold")

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/eda_physical.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 8 — Correlation Analysis
numeric_cols = [
    "f1_kd", "f2_kd", "f1_str_landed", "f2_str_landed",
    "f1_str_acc", "f2_str_acc", "f1_td_landed", "f2_td_landed",
    "f1_sub", "f2_sub", "round", "total_time_seconds", "f1_win"
]
corr_matrix = fights[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Feature Correlation Matrix", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{DATA_DIR}/eda_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 9 — Key Insights
print("=" * 60)
print("KEY STATISTICAL INSIGHTS")
print("=" * 60)

print(f"\n📊 DATASET OVERVIEW:")
print(f"   Total fights: {len(fights)}")
print(f"   Total events: {fights['event_name'].nunique()}")
print(f"   Total fighters: {len(fighters)}")
print(f"   Date range: {fights['event_date'].min()} to {fights['event_date'].max()}")

print(f"\n🥊 FIGHT OUTCOMES:")
print(f"   KO/TKO rate: {(fights['finish_type'] == 'KO/TKO').mean():.1%}")
print(f"   Submission rate: {(fights['finish_type'] == 'SUB').mean():.1%}")
print(f"   Decision rate: {(fights['finish_type'] == 'DEC').mean():.1%}")
print(f"   Average fight length: {fights['total_time_seconds'].mean()/60:.1f} minutes")

print(f"\n📈 STRIKING STATS:")
print(f"   Avg sig. strikes landed (winner): {fights['winner_str_landed'].mean():.1f}")
print(f"   Avg sig. strikes landed (loser): {fights['loser_str_landed'].mean():.1f}")
print(f"   Avg striking accuracy (winner): {fights['winner_str_acc'].mean():.1%}")
print(f"   Avg striking accuracy (loser): {fights['loser_str_acc'].mean():.1%}")

print(f"\n⚖️ POSITIONAL ANALYSIS (Red Corner = Favorite):")
print(f"   Fighter 1 (red corner) win rate: {fights['f1_win'].mean():.1%}")
print(f"   This suggests {'significant' if abs(fights['f1_win'].mean() - 0.5) > 0.05 else 'minimal'} positional bias")
print(f"   Red corner baseline will be our naive model to beat")

In [ ]:
# Cell 10 — Correlations & Data Quality
print("🔗 TOP CORRELATIONS WITH F1 WINNING:")
win_corr = corr_matrix["f1_win"].drop("f1_win").sort_values(key=abs, ascending=False)
for feat, corr in win_corr.head(8).items():
    direction = "↑" if corr > 0 else "↓"
    print(f"   {direction} {feat}: {corr:.3f}")

print(f"\n{'=' * 60}")
print("DATA QUALITY REPORT")
print("=" * 60)

print(f"\nMissing Value Percentages:")
for col in fights.columns:
    missing_pct = fights[col].isnull().mean() * 100
    if missing_pct > 0:
        print(f"   {col}: {missing_pct:.1f}%")

print(f"\nDraw/NC fights: {(fights['winner'] == 'Draw/NC').sum()}")
print(f"Weight classes found: {fights['weight_class'].nunique()}")
print(f"\nWeight class breakdown:")
print(fights["weight_class"].value_counts().to_string())